# GitHub Repository Recommender

Website gợi ý repository GitHub dựa trên **tài khoản GitHub** và **hành vi tương tác** của người dùng. Toàn bộ chạy trong notebook này (Flask + SQLite + HTML/JS thuần).

| Chức năng | Cách làm |
|---|---|
| Authentication | GitHub OAuth (`/login` → `/callback`) |
| Searching | GitHub Search API (`/api/search`) |
| Save | Lưu / bỏ lưu repo vào SQLite (`/api/save`) |
| Click log | Mỗi lần mở repo ghi vào bảng `clicks` (`/api/click`) |
| Recommendation | Hồ sơ sở thích từ starred + repo của bạn + saved + clicks, rồi chấm điểm repo ứng viên (`/api/recommend`) |

## Chuẩn bị (làm một lần)
1. Vào https://github.com/settings/developers → **New OAuth App**.
2. `Homepage URL`: `http://localhost:5000`
3. `Authorization callback URL`: `http://localhost:5000/callback`
4. Copy **Client ID** và tạo **Client Secret**, nhập khi chạy cell cấu hình bên dưới.
5. Chạy lần lượt các cell, rồi mở http://localhost:5000

In [1]:
# Cài thư viện (bỏ qua nếu đã có)
%pip install -q flask requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Cấu hình

In [2]:
import os, json, math, sqlite3, secrets, threading
from collections import Counter
from datetime import datetime, timedelta
from functools import wraps
from getpass import getpass

import requests
from flask import Flask, session, redirect, request, jsonify, render_template_string
from werkzeug.serving import make_server

HOST, PORT = "localhost", 5000
BASE_URL = f"http://{HOST}:{PORT}"
DB_PATH = "repo_recsys.db"

# Đọc từ biến môi trường, nếu chưa có thì hỏi trực tiếp
GITHUB_CLIENT_ID = os.getenv("GITHUB_CLIENT_ID") or input("GitHub Client ID: ").strip()
GITHUB_CLIENT_SECRET = os.getenv("GITHUB_CLIENT_SECRET") or getpass("GitHub Client Secret: ").strip()

## 2. Cơ sở dữ liệu (SQLite)

- `users`: tài khoản đăng nhập GitHub
- `saved`: repo người dùng lưu / đánh dấu
- `clicks`: mỗi lần người dùng bấm vào một repo (dùng làm tín hiệu cho gợi ý)

> Access token đang lưu dạng văn bản thường để demo. Khi triển khai thật hãy mã hoá hoặc không lưu.

In [3]:
def db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with db() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY, login TEXT, avatar TEXT, token TEXT, created_at TEXT);
        CREATE TABLE IF NOT EXISTS saved (
            user_id INTEGER, repo_id INTEGER, full_name TEXT, url TEXT, description TEXT,
            language TEXT, stars INTEGER, topics TEXT, saved_at TEXT,
            PRIMARY KEY (user_id, repo_id));
        CREATE TABLE IF NOT EXISTS clicks (
            id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER, repo_id INTEGER, full_name TEXT,
            url TEXT, language TEXT, topics TEXT, source TEXT, clicked_at TEXT);
        """)

init_db()

## 3. GitHub API và bộ gợi ý

**Hồ sơ sở thích** (`build_profile`) cộng trọng số cho ngôn ngữ và topic từ:
- repo đã star (×1) và repo của chính bạn (×1)
- repo đã lưu (×3)
- repo đã click (×2)

**Gợi ý** (`recommend`) tìm repo ứng viên theo các topic/ngôn ngữ nổi bật nhất, loại repo bạn đã biết, rồi chấm điểm:
`điểm = tổng trọng số topic trùng + trọng số ngôn ngữ + log10(stars)`. Mỗi repo kèm lý do gợi ý. Nếu người dùng chưa có dữ liệu, hệ thống trả về repo đang lên (cold start).

In [4]:
API = "https://api.github.com"

def gh(path, token=None, params=None):
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.get(API + path, headers=headers, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

def slim(r):
    return {
        "id": r["id"], "full_name": r["full_name"], "url": r["html_url"],
        "description": r.get("description") or "", "language": r.get("language") or "",
        "stars": r.get("stargazers_count", 0), "forks": r.get("forks_count", 0),
        "topics": r.get("topics") or [],
    }

def search_repos(q, token, sort="stars", per_page=20, page=1):
    data = gh("/search/repositories", token,
              {"q": q, "sort": sort, "order": "desc", "per_page": per_page, "page": page})
    return [slim(r) for r in data.get("items", [])]

def build_profile(user_id, token):
    langs, topics, known = Counter(), Counter(), set()

    def add(repo, w):
        if repo.get("language"):
            langs[repo["language"]] += w
        for t in repo.get("topics") or []:
            topics[t] += w

    for path in ("/user/starred", "/user/repos"):
        try:
            for r in gh(path, token, {"per_page": 50}):
                known.add(r["id"]); add(slim(r), 1)
        except requests.HTTPError:
            pass

    with db() as c:
        for row in c.execute("SELECT * FROM saved WHERE user_id=?", (user_id,)):
            known.add(row["repo_id"])
            add({"language": row["language"], "topics": json.loads(row["topics"] or "[]")}, 3)
        for row in c.execute("SELECT * FROM clicks WHERE user_id=?", (user_id,)):
            known.add(row["repo_id"])
            add({"language": row["language"], "topics": json.loads(row["topics"] or "[]")}, 2)
    return langs, topics, known

def recommend(user_id, token, limit=20):
    langs, topics, known = build_profile(user_id, token)

    if not langs and not topics:  # cold start: repo mới và đang lên
        since = (datetime.utcnow() - timedelta(days=30)).strftime("%Y-%m-%d")
        out = search_repos(f"created:>{since} stars:>50", token, per_page=limit)
        for r in out:
            r["reason"] = "Đang lên trong 30 ngày qua"
        return out

    candidates = {}
    queries = [f"topic:{t} stars:>100" for t, _ in topics.most_common(4)]
    queries += [f"language:{l} stars:>1000" for l, _ in langs.most_common(2)]
    for q in queries:
        try:
            for r in search_repos(q, token, per_page=15):
                if r["id"] not in known:
                    candidates[r["id"]] = r
        except requests.HTTPError:
            continue

    scored = []
    for r in candidates.values():
        matched = [t for t in r["topics"] if t in topics]
        score = sum(topics[t] for t in matched) + langs.get(r["language"], 0) \
                + math.log10(r["stars"] + 1)
        reasons = []
        if matched:
            reasons.append("Cùng chủ đề: " + ", ".join(matched[:3]))
        if r["language"] in langs:
            reasons.append(f"Bạn hay dùng {r['language']}")
        r["reason"] = "; ".join(reasons) or "Phổ biến trong lĩnh vực bạn quan tâm"
        scored.append((score, r))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for _, r in scored[:limit]]

## 4. Giao diện (một trang, HTML + JS thuần)

In [5]:
PAGE = """
<!doctype html>
<html lang="vi">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Repo Recommender</title>
<style>
  :root { --bg:#f3f5f0; --ink:#1c2321; --muted:#5d6b66; --line:#d5ddd7; --accent:#2f6f5e; --card:#fbfcfa; }
  @media (prefers-color-scheme: dark) {
    :root { --bg:#141a18; --ink:#e6ece8; --muted:#8fa099; --line:#2b3632; --accent:#6fc2a8; --card:#1a2220; }
  }
  * { box-sizing:border-box; }
  body { margin:0; background:var(--bg); color:var(--ink);
         font:16px/1.55 "Segoe UI", system-ui, sans-serif; }
  header { display:flex; align-items:center; justify-content:space-between; gap:12px;
           padding:14px max(16px, calc(50% - 400px)); border-bottom:1px solid var(--line); }
  h1 { font:600 1.15rem ui-monospace, Consolas, monospace; margin:0; }
  .who { display:flex; align-items:center; gap:10px; font-size:.9rem; }
  .who img { width:28px; height:28px; border-radius:50%; }
  main { max-width:800px; margin:0 auto; padding:20px 16px 60px; }
  a { color:var(--accent); }
  button, .btn { font:inherit; cursor:pointer; border:1px solid var(--line); background:var(--card);
                 color:var(--ink); padding:6px 12px; border-radius:6px; text-decoration:none; }
  button:hover, .btn:hover { border-color:var(--accent); }
  button:focus-visible, input:focus-visible, select:focus-visible, a:focus-visible
    { outline:2px solid var(--accent); outline-offset:2px; }
  .primary { background:var(--accent); border-color:var(--accent); color:#fff; }
  nav { display:flex; gap:4px; border-bottom:1px solid var(--line); margin-bottom:18px; }
  nav button { border:0; border-bottom:2px solid transparent; border-radius:0; background:none; padding:10px 14px; }
  nav button[aria-selected=true] { border-bottom-color:var(--accent); font-weight:600; }
  form { display:flex; gap:8px; flex-wrap:wrap; margin-bottom:16px; }
  input, select { font:inherit; padding:7px 10px; border:1px solid var(--line); border-radius:6px;
                  background:var(--card); color:var(--ink); }
  input[type=search] { flex:1; min-width:180px; }
  .repo { padding:14px 0; border-bottom:1px solid var(--line); }
  .repo h3 { margin:0 0 4px; font:600 1rem ui-monospace, Consolas, monospace; }
  .repo p { margin:0 0 6px; color:var(--muted); }
  .meta { display:flex; flex-wrap:wrap; gap:6px 14px; font-size:.85rem; color:var(--muted); align-items:center; }
  .reason { color:var(--accent); font-size:.85rem; margin-bottom:4px; }
  .tag { border:1px solid var(--line); border-radius:99px; padding:0 8px; font-size:.78rem; }
  .empty { color:var(--muted); padding:30px 0; text-align:center; }
  .hero { padding:60px 0; text-align:center; }
  .hero h2 { font-size:1.6rem; margin:0 0 8px; }
</style>
</head>
<body>
<header>
  <h1>repo-recommender</h1>
  <div class="who">
    {% if user %}
      <img src="{{ user.avatar }}" alt=""><span>{{ user.login }}</span>
      <a class="btn" href="/logout">Đăng xuất</a>
    {% else %}
      <a class="btn primary" href="/login">Đăng nhập bằng GitHub</a>
    {% endif %}
  </div>
</header>
<main>
{% if not user %}
  <div class="hero">
    <h2>Tìm repository hợp với bạn</h2>
    <p>Đăng nhập bằng GitHub để nhận gợi ý dựa trên repo bạn đã star, đã lưu và đã mở.</p>
    <a class="btn primary" href="/login">Đăng nhập bằng GitHub</a>
  </div>
{% else %}
  <nav role="tablist">
    <button role="tab" data-tab="recommend" aria-selected="true">Gợi ý cho bạn</button>
    <button role="tab" data-tab="search" aria-selected="false">Tìm kiếm</button>
    <button role="tab" data-tab="saved" aria-selected="false">Đã lưu</button>
    <button role="tab" data-tab="history" aria-selected="false">Đã xem</button>
  </nav>
  <form id="searchForm" hidden>
    <input type="search" id="q" placeholder="Ví dụ: machine learning, fastapi, ..." required>
    <select id="lang">
      <option value="">Mọi ngôn ngữ</option>
      <option>Python</option><option>JavaScript</option><option>TypeScript</option>
      <option>Java</option><option>Go</option><option>Rust</option><option>C++</option>
    </select>
    <select id="sort">
      <option value="stars">Nhiều sao nhất</option>
      <option value="updated">Mới cập nhật</option>
      <option value="forks">Nhiều fork</option>
    </select>
    <button class="primary" type="submit">Tìm</button>
  </form>
  <div id="list"></div>
{% endif %}
</main>
{% if user %}
<script>
const $ = s => document.querySelector(s);
const list = $("#list");
let savedIds = new Set();
let currentTab = "recommend";

const esc = s => String(s ?? "").replace(/[&<>"']/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
async function api(url, opts) {
  const r = await fetch(url, opts);
  if (!r.ok) throw new Error((await r.json().catch(() => ({}))).error || r.statusText);
  return r.json();
}
const post = (url, body) => api(url, {method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(body)});

function card(r, source) {
  const isSaved = savedIds.has(r.id);
  const el = document.createElement("article");
  el.className = "repo";
  el.innerHTML = `
    ${r.reason ? `<div class="reason">${esc(r.reason)}</div>` : ""}
    <h3><a href="${esc(r.url)}" target="_blank" rel="noopener">${esc(r.full_name)}</a></h3>
    <p>${esc(r.description) || "Chưa có mô tả"}</p>
    <div class="meta">
      ${r.language ? `<span>${esc(r.language)}</span>` : ""}
      <span>★ ${(r.stars ?? 0).toLocaleString()}</span>
      ${(r.topics || []).slice(0,4).map(t => `<span class="tag">${esc(t)}</span>`).join("")}
      <button type="button" class="save">${isSaved ? "Bỏ lưu" : "Lưu"}</button>
    </div>`;
  el.querySelector("h3 a").addEventListener("click", () => {
    fetch("/api/click", {method:"POST", keepalive:true,
      headers:{"Content-Type":"application/json"}, body:JSON.stringify({repo:r, source})});
  });
  const btn = el.querySelector(".save");
  btn.addEventListener("click", async () => {
    const want = !savedIds.has(r.id);
    try {
      await post(want ? "/api/save" : "/api/unsave", {repo:r});
      want ? savedIds.add(r.id) : savedIds.delete(r.id);
      btn.textContent = want ? "Bỏ lưu" : "Lưu";
      if (currentTab === "saved" && !want) el.remove();
    } catch (e) { alert(e.message); }
  });
  return el;
}

function render(items, source, emptyMsg) {
  list.innerHTML = "";
  if (!items.length) { list.innerHTML = `<div class="empty">${emptyMsg}</div>`; return; }
  items.forEach(r => list.appendChild(card(r, source)));
}
const loading = msg => list.innerHTML = `<div class="empty">${msg}</div>`;

async function load(tab) {
  currentTab = tab;
  document.querySelectorAll("nav button").forEach(b => b.setAttribute("aria-selected", b.dataset.tab === tab));
  $("#searchForm").hidden = tab !== "search";
  try {
    if (tab === "recommend") {
      loading("Đang phân tích sở thích của bạn…");
      render(await api("/api/recommend"), "recommend", "Chưa có gợi ý. Hãy thử tìm kiếm và lưu vài repo.");
    } else if (tab === "saved") {
      render(await api("/api/saved"), "saved", "Bạn chưa lưu repo nào. Bấm Lưu ở bất kỳ repo nào để thêm vào đây.");
    } else if (tab === "history") {
      render(await api("/api/history"), "history", "Chưa có repo nào được mở.");
    } else {
      list.innerHTML = `<div class="empty">Nhập từ khoá để tìm repository.</div>`;
    }
  } catch (e) { loading("Có lỗi: " + esc(e.message)); }
}

document.querySelectorAll("nav button").forEach(b => b.addEventListener("click", () => load(b.dataset.tab)));
$("#searchForm").addEventListener("submit", async e => {
  e.preventDefault();
  const p = new URLSearchParams({q:$("#q").value, language:$("#lang").value, sort:$("#sort").value});
  loading("Đang tìm…");
  try { render(await api("/api/search?" + p), "search", "Không tìm thấy repo phù hợp. Thử từ khoá khác."); }
  catch (err) { loading("Có lỗi: " + esc(err.message)); }
});

(async () => {
  try { (await api("/api/saved")).forEach(r => savedIds.add(r.id)); } catch (e) {}
  load("recommend");
})();
</script>
{% endif %}
</body>
</html>
"""

## 5. Server Flask

| Route | Việc làm |
|---|---|
| `GET /login`, `GET /callback`, `GET /logout` | Đăng nhập GitHub OAuth |
| `GET /api/search` | Tìm repo |
| `POST /api/save`, `POST /api/unsave`, `GET /api/saved` | Lưu / bỏ lưu / xem danh sách lưu |
| `POST /api/click`, `GET /api/history` | Ghi và xem lịch sử repo đã mở |
| `GET /api/recommend` | Gợi ý cho người dùng hiện tại |

In [6]:
app = Flask(__name__)
app.secret_key = secrets.token_hex(32)

def current_user():
    uid = session.get("uid")
    if not uid:
        return None
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE id=?", (uid,)).fetchone()
    return dict(row) if row else None

def login_required(fn):
    @wraps(fn)
    def wrapper(*a, **kw):
        user = current_user()
        if not user:
            return jsonify(error="Bạn cần đăng nhập"), 401
        return fn(user, *a, **kw)
    return wrapper

# ---------- Authentication ----------
@app.get("/")
def index():
    return render_template_string(PAGE, user=current_user())

@app.get("/login")
def login():
    session["state"] = secrets.token_urlsafe(16)
    params = {"client_id": GITHUB_CLIENT_ID, "redirect_uri": f"{BASE_URL}/callback",
              "scope": "read:user", "state": session["state"]}
    return redirect("https://github.com/login/oauth/authorize?" + requests.compat.urlencode(params))

@app.get("/callback")
def callback():
    if request.args.get("state") != session.pop("state", None):
        return "State không hợp lệ", 400
    r = requests.post("https://github.com/login/oauth/access_token",
                      headers={"Accept": "application/json"}, timeout=15,
                      data={"client_id": GITHUB_CLIENT_ID, "client_secret": GITHUB_CLIENT_SECRET,
                            "code": request.args.get("code"), "redirect_uri": f"{BASE_URL}/callback"})
    token = r.json().get("access_token")
    if not token:
        return "Đăng nhập thất bại: " + json.dumps(r.json()), 400
    me = gh("/user", token)
    with db() as c:
        c.execute("""INSERT INTO users (id, login, avatar, token, created_at) VALUES (?,?,?,?,?)
                     ON CONFLICT(id) DO UPDATE SET login=excluded.login, avatar=excluded.avatar, token=excluded.token""",
                  (me["id"], me["login"], me["avatar_url"], token, datetime.utcnow().isoformat()))
    session["uid"] = me["id"]
    return redirect("/")

@app.get("/logout")
def logout():
    session.clear()
    return redirect("/")

# ---------- Searching ----------
@app.get("/api/search")
@login_required
def api_search(user):
    q = request.args.get("q", "").strip()
    if not q:
        return jsonify([])
    if request.args.get("language"):
        q += f" language:{request.args['language']}"
    try:
        return jsonify(search_repos(q, user["token"], sort=request.args.get("sort", "stars")))
    except requests.HTTPError as e:
        return jsonify(error=f"GitHub API lỗi ({e.response.status_code}). Có thể bạn đã vượt giới hạn tìm kiếm, hãy thử lại sau ít phút."), 502

# ---------- Save / bookmark ----------
@app.post("/api/save")
@login_required
def api_save(user):
    r = request.json["repo"]
    with db() as c:
        c.execute("""INSERT OR REPLACE INTO saved
                     (user_id, repo_id, full_name, url, description, language, stars, topics, saved_at)
                     VALUES (?,?,?,?,?,?,?,?,?)""",
                  (user["id"], r["id"], r["full_name"], r["url"], r.get("description"), r.get("language"),
                   r.get("stars", 0), json.dumps(r.get("topics", [])), datetime.utcnow().isoformat()))
    return jsonify(ok=True)

@app.post("/api/unsave")
@login_required
def api_unsave(user):
    with db() as c:
        c.execute("DELETE FROM saved WHERE user_id=? AND repo_id=?", (user["id"], request.json["repo"]["id"]))
    return jsonify(ok=True)

def rows_to_repos(rows):
    return [{"id": r["repo_id"], "full_name": r["full_name"], "url": r["url"],
             "description": r["description"] if "description" in r.keys() else "",
             "language": r["language"], "stars": r["stars"] if "stars" in r.keys() else 0,
             "topics": json.loads(r["topics"] or "[]")} for r in rows]

@app.get("/api/saved")
@login_required
def api_saved(user):
    with db() as c:
        rows = c.execute("SELECT * FROM saved WHERE user_id=? ORDER BY saved_at DESC", (user["id"],)).fetchall()
    return jsonify(rows_to_repos(rows))

# ---------- Click tracking ----------
@app.post("/api/click")
@login_required
def api_click(user):
    r, source = request.json["repo"], request.json.get("source", "")
    with db() as c:
        c.execute("""INSERT INTO clicks (user_id, repo_id, full_name, url, language, topics, source, clicked_at)
                     VALUES (?,?,?,?,?,?,?,?)""",
                  (user["id"], r["id"], r["full_name"], r["url"], r.get("language"),
                   json.dumps(r.get("topics", [])), source, datetime.utcnow().isoformat()))
    return jsonify(ok=True)

@app.get("/api/history")
@login_required
def api_history(user):
    with db() as c:
        rows = c.execute("""SELECT repo_id, full_name, url, language, topics, MAX(clicked_at) AS t
                            FROM clicks WHERE user_id=? GROUP BY repo_id ORDER BY t DESC LIMIT 50""",
                         (user["id"],)).fetchall()
    return jsonify(rows_to_repos(rows))

# ---------- Recommendation ----------
@app.get("/api/recommend")
@login_required
def api_recommend(user):
    try:
        return jsonify(recommend(user["id"], user["token"]))
    except requests.HTTPError as e:
        return jsonify(error=f"GitHub API lỗi ({e.response.status_code}). Hãy thử lại sau ít phút."), 502

## 6. Chạy website

Server chạy ở nền nên notebook vẫn dùng được. Mở http://localhost:5000 sau khi chạy cell này. Muốn tắt server, chạy `stop_server()`.

In [7]:
_server = None

def start_server():
    global _server
    if _server:
        print("Server đang chạy")
        return
    _server = make_server(HOST, PORT, app, threaded=True)
    threading.Thread(target=_server.serve_forever, daemon=True).start()
    print(f"Website đang chạy tại {BASE_URL}")

def stop_server():
    global _server
    if _server:
        _server.shutdown()
        _server = None
        print("Đã dừng server")

start_server()

Website đang chạy tại http://localhost:5000


## 7. Xem dữ liệu tương tác

Chạy cell này bất cứ lúc nào để xem lượt click và repo đã lưu. Đây cũng là dữ liệu bạn có thể dùng để đánh giá chất lượng gợi ý trong báo cáo đồ án.

In [8]:
import pandas as pd
with db() as c:
    clicks = pd.read_sql("SELECT user_id, full_name, language, source, clicked_at FROM clicks ORDER BY id DESC", c)
    saved = pd.read_sql("SELECT user_id, full_name, language, stars, saved_at FROM saved ORDER BY saved_at DESC", c)
print("Lượt click:", len(clicks), "| Repo đã lưu:", len(saved))
display(clicks.head(10)); display(saved.head(10))

Lượt click: 0 | Repo đã lưu: 0


,user_id,full_name,language,source,clicked_at


,user_id,full_name,language,stars,saved_at


## Hướng mở rộng cho đồ án
- Thêm collaborative filtering: gợi ý repo mà những người dùng có hồ sơ giống bạn đã lưu.
- Dùng embedding của README hoặc mô tả (ví dụ sentence-transformers) thay cho so khớp topic.
- Giảm trọng số click cũ theo thời gian, và tách tín hiệu "click" khỏi "lưu" bằng trọng số học được.
- Đánh giá offline bằng Precision@K, Recall@K, NDCG trên bảng `clicks` và `saved`.
- Cache kết quả GitHub API để tránh vượt giới hạn tìm kiếm (30 request/phút khi đã đăng nhập).